In [1]:
import os, warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

warnings.filterwarnings("ignore")

from mangrove_kb.registry import RuleRegistry
from mangrove_kb.docstring_parser import parse_all_signals
from mangrove_kb.signals import momentum, trend, volatility, volume, patterns

# Use the docstring parser to extract metadata (replaces signals_metadata.json)
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "data")
METADATA = parse_all_signals([momentum, trend, volume, volatility, patterns])

EXCLUDE = set()  # No exclusions needed -- social signals are not in this package

def indicator_prefix(name):
    # "rsi_oversold" -> "rsi", "is_above_sma" -> "sma", "price_above_ema" -> "ema"
    special = {"is_above_sma": "sma", "price_above_ema": "ema",
               "mass_reversal_signal": "mass_index",
               "cumulative_return_positive": "cum_return",
               "cumulative_return_target": "cum_return",
               "daily_return_positive": "daily_return",
               "daily_return_negative": "daily_return",
               # Pattern signals
               "long_legged_doji_trigger": "doji",
               "dragonfly_doji_trigger": "doji",
               "gravestone_doji_trigger": "doji",
               "hanging_man_trigger": "hammer",
               "inverted_hammer_trigger": "hammer",
               "shooting_star_trigger": "hammer",
               "spinning_top_trigger": "spinning_top",
               "bullish_engulfing_trigger": "engulfing",
               "bearish_engulfing_trigger": "engulfing",
               "bullish_harami_trigger": "harami",
               "bearish_harami_trigger": "harami",
               "piercing_line_trigger": "piercing_line",
               "dark_cloud_cover_trigger": "dark_cloud",
               "morning_star_trigger": "morning_star",
               "evening_star_trigger": "evening_star",
               "three_white_soldiers_trigger": "3_candle",
               "three_black_crows_trigger": "3_candle",
               "three_inside_up_trigger": "3_candle",
               "three_inside_down_trigger": "3_candle",
               "inside_bar_trigger": "inside_bar",
               "outside_bar_trigger": "outside_bar",
               "bullish_pin_bar_trigger": "pin_bar",
               "bearish_pin_bar_trigger": "pin_bar",
               "two_bar_reversal_bullish_trigger": "2bar_rev",
               "two_bar_reversal_bearish_trigger": "2bar_rev",
               "bullish_pattern_recent": "pattern_filter",
               "bearish_pattern_recent": "pattern_filter",
               "reversal_pattern_bullish": "pattern_filter",
               "reversal_pattern_bearish": "pattern_filter",
               "continuation_pattern_bullish": "pattern_filter",
               "continuation_pattern_bearish": "pattern_filter",
               "indecision_pattern_recent": "pattern_filter",
               "strong_body_recent": "pattern_filter"}
    if name in special:
        return special[name]
    return name.split("_")[0]

TRIGGER_NAMES = sorted([k for k, v in METADATA.items()
                        if v.get("type") == "TRIGGER" and k not in EXCLUDE
                        and k in RuleRegistry._registry])
FILTER_NAMES  = sorted([k for k, v in METADATA.items()
                        if v.get("type") == "FILTER" and k not in EXCLUDE
                        and k in RuleRegistry._registry])

# Display labels: "rsi_oversold  [rsi]"
def make_label(name):
    return f"{name}  [{indicator_prefix(name)}]"

TRIGGER_OPTIONS = [(make_label(n), n) for n in TRIGGER_NAMES]
FILTER_OPTIONS  = [(make_label(n), n) for n in FILTER_NAMES]

print(f"Triggers: {len(TRIGGER_NAMES)}  |  Filters: {len(FILTER_NAMES)}")

ModuleNotFoundError: No module named 'ipywidgets'

In [2]:
DATASETS = {
    "BTC_1D":   "btc_2022-08-01_2026-02-15_1d.csv",
    "ETH_4H":   "eth_2024-01-01_2026-02-01_4h.csv",
    "PAXG_1H":  "paxg_2025-01-01_2026-02-14_1h.csv",
    "LINK_30M": "link_2025-07-04_2026-02-12_30m.csv",
    "XRP_15M":  "xrp_2025-12-20_2026-02-16_15m.csv",
    "SOL_5M":   "sol_2026-02-01_2026-02-16_5m.csv",
    "DOGE_5M":  "doge_2021-04-01_2021-06-15_5m.csv",
}
COLUMN_MAP = {"open": "Open", "high": "High", "low": "Low",
              "close": "Close", "volume": "Volume"}
_df_cache = {}

def load_dataset(key):
    if key in _df_cache:
        return _df_cache[key]
    path = os.path.join(DATA_DIR, DATASETS[key])
    df = pd.read_csv(path)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.rename(columns=COLUMN_MAP).sort_values("timestamp").reset_index(drop=True)
    _df_cache[key] = df
    return df

In [3]:
def make_param_widget(pname, pspec):
    ptype   = pspec.get("type", "int")
    default = pspec.get("default")
    label   = widgets.Label(value=f"{pname}:",
                            layout=widgets.Layout(width="130px"))
    if ptype == "bool":
        val = bool(default) if default is not None else False
        w = widgets.Checkbox(value=val, description="", indent=False,
                             layout=widgets.Layout(width="60px"))
    elif ptype == "str":
        options = ["bullish", "bearish"]
        val = default if default in options else options[0]
        w = widgets.Dropdown(options=options, value=val,
                             layout=widgets.Layout(width="120px"))
    elif ptype == "float":
        lo   = float(pspec.get("min", 0))
        hi   = float(pspec.get("max", 100))
        val  = float(default) if default is not None else (lo + hi) / 2
        val  = max(lo, min(hi, val))
        step = round((hi - lo) / 100, 4) or 0.01
        w = widgets.FloatSlider(value=val, min=lo, max=hi, step=step,
                                continuous_update=False, readout=True,
                                readout_format=".2f",
                                layout=widgets.Layout(width="240px"))
    else:
        lo  = int(pspec.get("min", 1))
        hi  = int(pspec.get("max", 200))
        val = int(default) if default is not None else (lo + hi) // 2
        val = max(lo, min(hi, val))
        w = widgets.IntSlider(value=val, min=lo, max=hi, step=1,
                              continuous_update=False,
                              layout=widgets.Layout(width="240px"))
    return w, widgets.HBox([label, w])


def build_signal_panel(signal_name):
    meta  = METADATA.get(signal_name, {})
    stype = meta.get("type", "?")
    ind   = indicator_prefix(signal_name)
    header = widgets.HTML(
        value=(f"<b style='font-size:0.9em'>{signal_name}</b> "
               f"<span style='color:#888;font-size:0.8em'>[{stype} | {ind}]</span>")
    )
    param_widgets = {}
    rows = [header]
    for pname, pspec in meta.get("params", {}).items():
        w, row = make_param_widget(pname, pspec)
        param_widgets[pname] = w
        rows.append(row)
    box = widgets.VBox(rows, layout=widgets.Layout(
        border="1px solid #ddd", padding="6px", margin="3px 0px",
        border_radius="4px", min_width="400px"
    ))
    return box, param_widgets

In [4]:
# Palette for per-filter shading (up to 8 filters at once)
_FILTER_COLORS = [
    "#4c9be8",  # blue
    "#e88c4c",  # orange
    "#7ecf7e",  # green
    "#c47ece",  # purple
    "#e8d24c",  # yellow
    "#e84c7e",  # pink
    "#4ce8cc",  # teal
    "#a07060",  # brown
]

# -- Create the FigureWidget once and display it immediately ------------------
_fig = go.FigureWidget(layout=go.Layout(
    title=dict(
        text="Signal Explorer -- BTC 1D",
        font=dict(size=15, color="#1a1a2e"),
        x=0.01, xanchor="left",
    ),
    xaxis=dict(
        title=dict(text="Date / Time", font=dict(size=11)),
        rangeslider=dict(visible=False),
        showgrid=True, gridcolor="#e8e8e8",
        type="date",
    ),
    yaxis=dict(
        title=dict(text="Price (USD)", font=dict(size=11)),
        showgrid=True, gridcolor="#e8e8e8",
        tickformat=",.4f",
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom", y=1.03,
        xanchor="left", x=0,
        font=dict(size=10),
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="#cccccc", borderwidth=1,
    ),
    height=560,
    autosize=True,
    margin=dict(l=70, r=30, t=80, b=55),
    plot_bgcolor="#fafafa",
    paper_bgcolor="#ffffff",
    hovermode="x unified",
))
_fig.add_trace(go.Scatter(
    x=[], y=[], mode="lines",
    line=dict(color="#1a1a2e", width=1.2),
    name="Close",
    hovertemplate="<b>%{x|%Y-%m-%d %H:%M}</b><br>Close: %{y:,.4f}<extra></extra>",
))

# -- Controls -----------------------------------------------------------------
dataset_dd = widgets.Dropdown(
    options=list(DATASETS.keys()), value="BTC_1D",
    description="Dataset:", layout=widgets.Layout(width="215px"),
)
warmup_spin = widgets.BoundedIntText(
    value=20, min=1, max=500, description="Warmup:",
    layout=widgets.Layout(width="175px"),
)
triggers_select = widgets.SelectMultiple(
    options=TRIGGER_OPTIONS, rows=12, description="",
    layout=widgets.Layout(width="290px", height="220px"),
)
filters_select = widgets.SelectMultiple(
    options=FILTER_OPTIONS, rows=12, description="",
    layout=widgets.Layout(width="290px", height="220px"),
)
run_btn    = widgets.Button(description="Run signals", button_style="primary",
                            layout=widgets.Layout(width="110px"))
status_lbl = widgets.Label(value="")
params_area  = widgets.VBox([], layout=widgets.Layout(min_width="420px"))
toggles_area = widgets.VBox([])

# -- State --------------------------------------------------------------------
_param_widgets  = {}
_signal_hits    = {}
_signal_types   = {}
_toggle_widgets = {}
_current_df     = None

# -- Price refresh (called on dataset change and at init) ---------------------
def refresh_price(key):
    global _current_df
    df = load_dataset(key)
    _current_df = df
    # Build a nice title: "BTC_1D" -> "Signal Explorer -- BTC 1D"
    pretty = key.replace("_", " ")
    with _fig.batch_update():
        _fig.data[0].x = df["timestamp"]
        _fig.data[0].y = df["Close"]
        _fig.layout.title.text = f"Signal Explorer -- {pretty}"
        _fig.layout.xaxis.title.text = "Date / Time"
        _fig.layout.yaxis.title.text = "Price (USD)"
        while len(_fig.data) > 1:
            _fig.data = _fig.data[:1]
        _fig.layout.shapes = []
    _signal_hits.clear()
    _signal_types.clear()
    _toggle_widgets.clear()
    toggles_area.children = []
    status_lbl.value = ""

def on_dataset_change(change):
    refresh_price(change["new"])

dataset_dd.observe(on_dataset_change, names="value")

# -- Param panels: rebuild when either signal list changes --------------------
def rebuild_param_panels(*_):
    selected_t = list(triggers_select.value)
    selected_f = list(filters_select.value)
    all_selected = selected_t + selected_f
    panels, new_pw = [], {}
    for name in all_selected:
        panel, pw = build_signal_panel(name)
        if name in _param_widgets:
            for pname, old_w in _param_widgets[name].items():
                if pname in pw:
                    try:
                        pw[pname].value = old_w.value
                    except Exception:
                        pass
        new_pw[name] = pw
        panels.append(panel)
    _param_widgets.clear()
    _param_widgets.update(new_pw)
    params_area.children = panels

triggers_select.observe(rebuild_param_panels, names="value")
filters_select.observe(rebuild_param_panels, names="value")

# -- Build contiguous span index list from a boolean mask --------------------
def _spans_from_mask(arr):
    hit_idx = np.where(arr)[0]
    if len(hit_idx) == 0:
        return []
    spans, s, p = [], hit_idx[0], hit_idx[0]
    for h in hit_idx[1:]:
        if h != p + 1:
            spans.append((s, p))
            s = h
        p = h
    spans.append((s, p))
    return spans

# -- Build a filled scatter trace for one filter band -----------------------
# Uses "toself" fill covering [y_lo, y_hi] for each contiguous active span.
def _filter_band_trace(df, mask, color, label):
    n = len(df)
    spans = _spans_from_mask(mask)
    if not spans:
        return None
    # Compute band bounds: 2% of price range above/below close midpoint
    price_lo = float(df["Low"].min())
    price_hi = float(df["High"].max())
    band_half = (price_hi - price_lo) * 0.018   # 1.8% half-width
    mid       = (price_hi + price_lo) / 2.0
    band_lo   = mid - band_half * 14             # wide enough to be visible
    band_hi   = mid + band_half * 14
    # Use full y-range (paper) via shapes instead for cleaner look;
    # but for legend we add a zero-size scatter for the color swatch.
    # Build shapes list for this filter:
    shapes = []
    for (s0, s1) in spans:
        shapes.append(dict(
            type="rect",
            x0=str(df["timestamp"].iloc[s0]),
            x1=str(df["timestamp"].iloc[min(s1 + 1, n - 1)]),
            y0=0, y1=1, xref="x", yref="paper",
            fillcolor=color, opacity=0.18,
            line_width=0, layer="below",
        ))
    count = int(mask.sum())
    # Dummy scatter trace for legend swatch
    dummy = go.Scatter(
        x=[df["timestamp"].iloc[spans[0][0]]],
        y=[float(df["Close"].iloc[spans[0][0]])],
        mode="markers",
        marker=dict(size=0, color=color, opacity=0),
        fill=None,
        showlegend=True,
        name=f"{label}  ({count} bars)",
        hoverinfo="skip",
    )
    return shapes, dummy

# -- AND logic: only active signals -------------------------------------------
def compute_combined():
    if _current_df is None:
        return np.array([], dtype=bool)
    n      = len(_current_df)
    active = [nm for nm, cb in _toggle_widgets.items() if cb.value]
    if not active or not _signal_hits:
        return np.zeros(n, dtype=bool)
    mask = np.ones(n, dtype=bool)
    for nm in active:
        if nm in _signal_hits:
            mask &= _signal_hits[nm]
    return mask

# -- Redraw signal overlay on figure ------------------------------------------
def update_signals():
    if _current_df is None:
        return
    df      = _current_df
    n       = len(df)
    active  = [nm for nm, cb in _toggle_widgets.items() if cb.value]

    with _fig.batch_update():
        # Reset to price-only
        _fig.data   = _fig.data[:1]
        _fig.layout.shapes = []

        if not active:
            return

        # --- 1. Draw per-filter shading (each active FILTER independently) ---
        all_shapes  = []
        filter_idx  = 0
        for nm in active:
            if _signal_types.get(nm) != "FILTER":
                continue
            color  = _FILTER_COLORS[filter_idx % len(_FILTER_COLORS)]
            filter_idx += 1
            mask   = _signal_hits.get(nm, np.zeros(n, dtype=bool))
            result = _filter_band_trace(df, mask, color, nm)
            if result is None:
                continue
            shapes, dummy = result
            all_shapes.extend(shapes)
            _fig.add_trace(dummy)
        _fig.layout.shapes = all_shapes

        # --- 2. Draw combined AND markers only when a TRIGGER is active ------
        has_trigger = any(_signal_types.get(nm) == "TRIGGER" for nm in active)
        if not has_trigger:
            return

        combined = compute_combined()
        hit_idx  = np.where(combined)[0]
        if len(hit_idx) == 0:
            return

        _fig.add_trace(go.Scatter(
            x=df["timestamp"].iloc[hit_idx],
            y=df["Close"].iloc[hit_idx],
            mode="markers",
            marker=dict(
                symbol="triangle-up", size=10, color="#2ecc71",
                line=dict(width=1, color="#1a9950"),
            ),
            name=f"Combined signal ({len(hit_idx)} hits)",
            hovertemplate="<b>%{x|%Y-%m-%d %H:%M}</b><br>Entry: %{y:,.4f}<extra></extra>",
        ))

def on_toggle_change(change):
    update_signals()

# -- Run ----------------------------------------------------------------------
def on_run(b):
    selected_t = list(triggers_select.value)
    selected_f = list(filters_select.value)
    all_selected = selected_t + selected_f
    if not all_selected:
        status_lbl.value = "Select at least one signal."
        return

    run_btn.disabled = True
    status_lbl.value = f"Evaluating {len(all_selected)} signal(s)..."
    toggles_area.children = []

    df     = load_dataset(dataset_dd.value)
    n      = len(df)
    warmup = warmup_spin.value

    _signal_hits.clear()
    _signal_types.clear()
    _toggle_widgets.clear()

    for sig_name in all_selected:
        pw     = _param_widgets.get(sig_name, {})
        params = {pname: w.value for pname, w in pw.items()}
        rule   = {"name": sig_name, "params": params}
        stype  = METADATA.get(sig_name, {}).get("type", "TRIGGER")
        arr    = np.zeros(n, dtype=bool)
        for i in range(warmup, n):
            try:
                if RuleRegistry.evaluate(rule, df.iloc[:i + 1]):
                    arr[i] = True
            except Exception as e:
                status_lbl.value = f"Error in {sig_name} at bar {i}: {e}"
                run_btn.disabled = False
                return
        _signal_hits[sig_name]  = arr
        _signal_types[sig_name] = stype

    # Build toggle checkboxes
    boxes = []
    for sig_name in all_selected:
        stype = _signal_types[sig_name]
        ind   = indicator_prefix(sig_name)
        count = int(_signal_hits[sig_name].sum())
        cb = widgets.Checkbox(
            value=True,
            description=f"{sig_name}  [{stype} | {ind}]  ({count} hits)",
            indent=False,
            layout=widgets.Layout(width="auto"),
            style={"description_width": "initial"},
        )
        cb.observe(on_toggle_change, names="value")
        _toggle_widgets[sig_name] = cb
        boxes.append(cb)
    toggles_area.children = boxes

    update_signals()
    summary = ", ".join(f"{sn}: {int(_signal_hits[sn].sum())}" for sn in all_selected)
    status_lbl.value = f"Done. {summary}"
    run_btn.disabled = False

run_btn.on_click(on_run)

# -- Initial price load -------------------------------------------------------
refresh_price("BTC_1D")

# -- Layout -------------------------------------------------------------------
trigger_col = widgets.VBox([
    widgets.HTML("<b style='font-size:0.85em'>TRIGGER signals</b>"),
    triggers_select,
])
filter_col = widgets.VBox([
    widgets.HTML("<b style='font-size:0.85em'>FILTER signals</b>"),
    filters_select,
])
left_col = widgets.VBox(
    [dataset_dd, warmup_spin, widgets.HBox([trigger_col, filter_col])],
    layout=widgets.Layout(gap="6px"),
)
top_row  = widgets.HBox([left_col, params_area],
                         layout=widgets.Layout(align_items="flex-start", gap="16px"))
btn_row  = widgets.HBox([run_btn, status_lbl],
                         layout=widgets.Layout(align_items="center", gap="8px"))

display(widgets.VBox([top_row, btn_row, toggles_area],
                     layout=widgets.Layout(gap="6px")))
display(_fig)


FigureWidget({
    'data': [{'hovertemplate': '<b>%{x|%Y-%m-%d %H:%M}</b><br>Close: %{y:,.4f}<extra></extra>',
              'line': {'color': '#1a1a2e', 'width': 1.2},
              'mode': 'lines',
              'name': 'Close',
              'type': 'scatter',
              'uid': '68629718-d666-4332-a055-57dfd715414d',
              'x': array(['2022-08-01T00:00:00.000000', '2022-08-02T00:00:00.000000',
                          '2022-08-03T00:00:00.000000', ..., '2026-02-12T00:00:00.000000',
                          '2026-02-13T00:00:00.000000', '2026-02-14T00:00:00.000000'],
                         shape=(1294,), dtype='datetime64[us]'),
              'y': {'bdata': ('PQrXowC51kD2KFyP8nLWQOF6FK6XSN' ... 'ejAi7wQMP1KFxfz/BA7FG4Hu8L8UA='),
                    'dtype': 'f8'}}],
    'layout': {'autosize': True,
               'height': 560,
               'hovermode': 'x unified',
               'legend': {'bgcolor': 'rgba(255,255,255,0.85)',
                          'bordercolor'